# 04 · Race simulator, Part A: single car and strategy search

Spain 2022 (dry, no safety car, not a case study). Verstappen from the end of
lap 1, alone against the clock (no rivals or safety cars yet: Part B).
Logic is in `f1strat.simulator`; see `docs/simulator.md`.

Driver pace comes from the full-data fit, so it includes hindsight about this
race. That is fine for a demo; Phase 6 uses cutoff fits.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

from f1strat.clean import clean_laps
from f1strat.data import load_season
from f1strat.deg_model import THIN_FIT_PATH, load_fit
from f1strat.simulator import (RaceState, actual_plan, car_model, make_draws,
                               observed_pit_losses, plan_totals, search)

FIG_DIR = Path("../reports/figures")
fitted = load_fit(THIN_FIT_PATH)
raw = load_season(2022)
cleaned = clean_laps(raw)
state = RaceState.from_laps(raw, 6, lap=1, total_laps=66)
car = state.cars["VER"]
car

CarState(driver='VER', team='Red Bull Racing', position=2, race_time=89.739, compound='SOFT', tyre_life=4, laps_since_stop=0, stint=1, compounds_used=frozenset({'SOFT'}))

In [2]:
rng = np.random.default_rng(42)
n_sims = 1000
model = car_model(fitted, 6, "VER", car.team, rng.permutation(1000)[:n_sims], rng,
                  current_stint=None)   # from lap 1 the first stint's effect is unknown
losses = observed_pit_losses(cleaned, 6)
draws = make_draws(model, state.laps_remaining, losses, max_stops=3, rng=rng)
print(f"{len(losses)} observed green-flag stops at Spain: median {np.median(losses):.1f} s, "
      f"max {losses.max():.1f} s")
actual = {d: actual_plan(raw, 6, d, 1) for d in ["VER", "PER", "RUS", "SAI", "BOT", "HAM"]}

51 observed green-flag stops at Spain: median 22.7 s, max 37.6 s


## Ranked plans

Each table lists the 10 best plans by mean race time, then the plans the
front-runners actually ran (applied to Verstappen's car). `saved_vs_ref_s` and
`p_beats_ref` compare each plan with Verstappen's actual plan in the same
simulation (common random numbers).

1. **Unlimited sets:** every compound available new (TyreLife 1) and used
   (TyreLife 4), without limit.
2. **Realistic sets:** new mediums and hards plus used softs, which is what the
   front-runners actually fitted. The real allocation is not in the data.

In [3]:
cols = ["plan", "stops", "mean_s", "gap_to_best_s", "p10_s", "p90_s",
        "saved_vs_ref_s", "saved_p10_s", "saved_p90_s", "p_beats_ref", "actual_by"]
tables = {}
for name, sets in [("unlimited", None),
                   ("realistic", [("MEDIUM", 1), ("HARD", 1), ("SOFT", 4)])]:
    start = time.time()
    res = search(model, state, "VER", draws, max_stops=3, shortlist=50,
                 include=list(actual.values()), allowed_sets=sets)
    elapsed = time.time() - start
    tab = res.table(reference=actual["VER"])
    tab["actual_by"] = tab["plan"].map(
        lambda l: ",".join(d for d, p in actual.items() if p.label("SOFT") == l))
    shown = pd.concat([tab.head(10), tab[tab["actual_by"] != ""]])[cols]
    shown.round(2).to_csv(FIG_DIR / f"spain_demo_{name}.csv", index=False)
    (FIG_DIR / f"spain_demo_{name}.md").write_text(shown.round(2).to_markdown(index=False) + "\n")
    tables[name] = shown
    print(f"{name}: {res.n_legal:,} legal plans scored, search + Monte Carlo {elapsed:.2f} s")
tables["unlimited"].round(2)

unlimited: 8,730,880 legal plans scored, search + Monte Carlo 0.88 s
realistic: 1,099,520 legal plans scored, search + Monte Carlo 0.14 s


,plan,stops,mean_s,gap_to_best_s,p10_s,p90_s,saved_vs_ref_s,saved_p10_s,saved_p90_s,p_beats_ref,actual_by
0,S → 15 S(new) → 33 S(new) → 51 M(new),3,5806.34,0.00,5788.43,5824.31,19.98,14.84,25.11,1.00,
1,S → 15 S(new) → 33 S(new) → 52 M(new),3,5806.36,0.02,5788.45,5824.52,19.96,14.36,25.53,1.00,
2,S → 16 S(new) → 34 S(new) → 52 M(new),3,5806.37,0.03,5788.68,5824.34,19.95,14.40,25.63,1.00,
3,S → 15 S(new) → 34 S(new) → 52 M(new),3,5806.38,0.04,5788.63,5824.48,19.94,14.37,25.52,1.00,
4,S → 15 S(new) → 32 S(new) → 51 M(new),3,5806.46,0.12,5788.59,5824.67,19.86,14.68,24.94,1.00,
5,S → 16 S(new) → 33 S(new) → 51 M(new),3,5806.47,0.13,5788.61,5824.57,19.85,14.75,25.01,1.00,
6,S → 14 S(new) → 32 S(new) → 51 M(new),3,5806.47,0.13,5788.32,5824.60,19.85,14.69,25.05,1.00,
7,S → 16 S(new) → 34 S(new) → 51 M(new),3,5806.48,0.15,5788.52,5824.43,19.84,14.73,25.04,1.00,
8,S → 14 S(new) → 33 S(new) → 51 M(new),3,5806.49,0.15,5788.50,5824.48,19.83,14.64,24.91,1.00,
9,S → 16 S(new) → 33 S(new) → 52 M(new),3,5806.49,0.15,5788.97,5824.48,19.83,14.13,25.50,1.00,


In [4]:
tables["realistic"].round(2)

,plan,stops,mean_s,gap_to_best_s,p10_s,p90_s,saved_vs_ref_s,saved_p10_s,saved_p90_s,p_beats_ref,actual_by
0,S → 17 M(new) → 33 S(used) → 50 S(used),3,5820.96,0.00,5803.35,5838.77,5.36,0.80,10.19,0.94,
1,S → 17 S(used) → 33 S(used) → 50 M(new),3,5820.96,0.00,5803.35,5838.77,5.36,0.80,10.19,0.94,
2,S → 16 S(used) → 33 S(used) → 50 M(new),3,5820.97,0.01,5803.23,5838.83,5.35,0.75,10.05,0.95,
3,S → 17 S(used) → 33 M(new) → 50 S(used),3,5820.97,0.01,5803.40,5838.84,5.35,0.91,10.09,0.95,
4,S → 17 S(used) → 34 M(new) → 50 S(used),3,5820.97,0.02,5803.57,5838.72,5.35,0.83,10.16,0.94,
5,S → 17 S(used) → 34 S(used) → 50 M(new),3,5820.97,0.02,5803.57,5838.72,5.35,0.83,10.16,0.94,
6,S → 16 M(new) → 33 S(used) → 50 S(used),3,5820.98,0.02,5803.35,5838.88,5.34,0.89,9.90,0.95,
7,S → 16 S(used) → 33 M(new) → 50 S(used),3,5820.98,0.02,5803.35,5838.88,5.34,0.89,9.90,0.95,
8,S → 17 M(new) → 34 S(used) → 50 S(used),3,5820.99,0.03,5803.66,5838.74,5.33,0.89,10.01,0.94,
9,S → 17 S(used) → 34 S(used) → 51 M(new),3,5821.01,0.06,5803.52,5838.87,5.31,0.26,10.56,0.92,


## Runtime: 50 plans × 1,000 simulations (Monte Carlo only)

In [5]:
start = time.time()
plan_totals(model, state, "VER", res.plans[:50], draws)
print(f"{(time.time() - start) * 1000:.1f} ms")

4.2 ms
